In [0]:
from pyspark.sql import Row

data = [
    Row(order_id=1, country="India", year=2024, amount=100),
    Row(order_id=2, country="India", year=2024, amount=200),
    Row(order_id=3, country="India", year=2025, amount=150),
    Row(order_id=4, country="US", year=2024, amount=300),
    Row(order_id=5, country="US", year=2025, amount=400),
    Row(order_id=6, country="UK", year=2025, amount=250),
]

df = spark.createDataFrame(data)
df.display()


In [0]:
df.explain(True)


In [0]:
from pyspark.sql.functions import spark_partition_id

df_with_pid = df.withColumn("partition_id", spark_partition_id())
df_with_pid.display()

In [0]:
base_path = "/Volumes/dataeng/source/data"

df.write.mode("overwrite").parquet(base_path)

In [0]:
df_with_pid.groupBy("partition_id").count().display()


In [0]:
from pyspark.sql import functions as F

df_big = (
    spark.range(0, 5_000_000)   # 5 million rows
    .withColumn("col1", F.repeat(F.lit("spark_demo_"), 20))
    .withColumn("col2", F.repeat(F.lit("databricks_"), 20))
    .withColumn("col3", F.rand())
    .withColumn("col4", F.rand())
)


In [0]:
from pyspark.sql.functions import spark_partition_id

df_big \
  .withColumn("partition_id", spark_partition_id()) \
  .groupBy("partition_id") \
  .count() \
  .display()


In [0]:
base_path = "/Volumes/dataeng/source/data"

df_big.write.mode("overwrite").parquet(base_path)

**RDD**

In [0]:
# RDD creation
rdd = spark.sparkContext.parallelize([
    ("India", 120),
    ("India", 90),
    ("US", 200),
    ("US", 50),
    ("UK", 300)
])

# RDD transformations
result_rdd = (
    rdd
    .filter(lambda x: x[1] > 100)
    .map(lambda x: (x[0], x[1]))
    .reduceByKey(lambda a, b: a + b)
)

result_rdd.collect()


**Dataframe**

In [0]:
from pyspark.sql import functions as F

df = spark.createDataFrame(
    [("India", 120), ("India", 90), ("US", 200), ("US", 50), ("UK", 300)],
    ["country", "amount"]
)

result_df = (
    df
    .filter(F.col("amount") > 100)
    .groupBy("country")
    .agg(F.sum("amount").alias("total_amount"))
)

result_df.display()
